# Session 1: every measurement has an error
**A. Real-world problem.** You measure a table with a school ruler and read 12.3 cm. Is it
exactly 12.3? No. The honest statement is 12.3 +- 0.1 cm: the true length is probably
between 12.2 and 12.4. When you put two such lengths into the formula for area, the
answer carries an error too. How big?

**Plan.** (1) write a measurement as value +- error, (2) find the smallest and largest
possible area, (3) compare the relative errors, (4) plot it.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: a measurement is an interval
"12.3 +- 0.1" means the true value is believed to lie in [12.2, 12.4]. The error 0.1 is
about half the smallest ruler mark. The *relative error* is error / value, usually as a
percentage: 0.1 / 12.3 = 0.81 percent.

In [ ]:
def interval(value, err):
    """Return (low, high) of a measurement value +- err."""
    return value - err, value + err


def relative_error(value, err):
    """Error as a fraction of the value (multiply by 100 for percent)."""
    return err / abs(value)


def half_range(readings):
    """Half the spread of repeated readings: a first, rough estimate of the error."""
    r = np.asarray(readings, dtype=float)
    return float((r.max() - r.min()) / 2.0)

## Step 2: carry the interval through the area formula
Area = length * width. The smallest area uses both smallest lengths; the largest uses both
largest lengths (all lengths are positive). This "worst case" is easy to see and a little
pessimistic: it assumes both errors push the same way at once.

In [ ]:
def rect_area(length, width):
    return length * width


def area_extremes(length, dl, width, dw):
    """Smallest and largest area when length and width are only known to +- dl and +- dw."""
    return rect_area(length - dl, width - dw), rect_area(length + dl, width + dw)

## Step 3: see it
Left: the measured rectangle with its smallest and largest versions. Right: area against
length; the band is the range of area for a length known only to +- 0.1 cm.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    L, dL, W, dW = 12.3, 0.1, 8.4, 0.1
    lo, hi = area_extremes(L, dL, W, dW)
    area = rect_area(L, W)
    print(f"area = {area:.2f} cm^2, smallest {lo:.2f}, largest {hi:.2f}, half-range {(hi - lo) / 2:.2f}")
    print(f"relative errors: length {100 * relative_error(L, dL):.2f}%, width {100 * relative_error(W, dW):.2f}%, "
          f"area {100 * (hi - lo) / 2 / area:.2f}%")

    fig, ax = plt.subplots(figsize=(6, 4.5))
    for (l, w), col, name in [((L + dL, W + dW), "tab:red", "largest"), ((L, W), "tab:blue", "measured"),
                              ((L - dL, W - dW), "tab:green", "smallest")]:
        ax.add_patch(plt.Rectangle((0, 0), l, w, fill=False, edgecolor=col, lw=2, label=f"{name}: {l:.1f} x {w:.1f}"))
    ax.set_xlim(11.5, 12.6)
    ax.set_ylim(7.8, 8.7)
    ax.set_aspect("equal")
    ax.set_xlabel("length (cm)")
    ax.set_ylabel("width (cm)")
    ax.set_title("Top-right corners: three possible rectangles")
    ax.legend(loc="lower left")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "rectangles.png", dpi=120)

    ls = np.linspace(11.8, 12.8, 101)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(ls, rect_area(ls, W), label="area at width 8.4 cm")
    ax2.axvspan(L - dL, L + dL, alpha=0.2, color="tab:orange", label="length 12.3 +- 0.1 cm")
    ax2.axhspan(rect_area(L - dL, W), rect_area(L + dL, W), alpha=0.2, color="tab:orange")
    ax2.set_xlabel("length (cm)")
    ax2.set_ylabel("area (cm^2)")
    ax2.set_title("A small length error becomes an area error")
    ax2.grid(alpha=0.3)
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "area_band.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Change the lengths and errors, predict the area error before you run, then answer the
questions in the README (section F) and run the checkpoint.

## Coding tasks
1. **Easy.** Change the lengths in `lesson.py` to 20.0 and 5.0 cm and predict the area range before running it; compare with the answer to question B in Section F.
2. **Stretch.** Write `volume_extremes(L, dL, W, dW, H, dH)` for a box and check that its relative error is about the sum of the three relative errors.
3. **Explain.** Explain why a ruler with half the error halves the area error (when the other side stays exact) but not necessarily the typical error of a long measurement chain.

## Assessment
- **A. Conceptual.** The worst-case area error for the table is 2.07 cm^2, but the typical error is smaller (about 1.49 cm^2, shown in Session 2). Why is the typical error smaller than the worst case?
- **B. Numerical.** A rectangle measures 20.0 +- 0.1 cm by 5.0 +- 0.1 cm. Find the area, its smallest and largest values, the worst-case error and the relative error.
- **C. Graph and simulation interpretation.** On the area graph (width 8.4 cm), how tall is the band at a length of 12.3 cm when the length error is 0.1 cm and the width is exact? Why is it a straight band?
- **D. Design and optimisation challenge.** Design: you may buy a finer ruler for only ONE of the two sides (halving its error to 0.05 cm). Which side should it be for the 12.3 by 8.4 cm table, and what is the new area error?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    lo, hi = L.interval(12.3, 0.1)
    checks.append(("12.3 +- 0.1 is the interval 12.2 to 12.4", abs(lo - 12.2) < 1e-12 and abs(hi - 12.4) < 1e-12))
    area = L.rect_area(12.3, 8.4)
    amin, amax = L.area_extremes(12.3, 0.1, 8.4, 0.1)
    checks.append(("worked example: area 103.32, smallest 101.26, largest 105.40 cm^2",
                   abs(area - 103.32) < 5e-3 and abs(amin - 101.26) < 5e-3 and abs(amax - 105.40) < 5e-3))
    checks.append(("worked example: area error is about +-2.07 cm^2 (half-range)", abs((amax - amin) / 2 - 2.07) < 5e-3))
    rl, rw = L.relative_error(12.3, 0.1), L.relative_error(8.4, 0.1)
    checks.append(("relative errors are 0.81 % and 1.19 %", abs(100 * rl - 0.81) < 0.005 and abs(100 * rw - 1.19) < 0.005))
    checks.append(("relative error of the area is about the sum 2.0 % (hand calculation)",
                   abs((amax - amin) / 2 / area - (rl + rw)) < 5e-4 and abs(100 * (rl + rw) - 2.0) < 0.01))
    checks.append(("half-range of readings 12.2, 12.4, 12.3, 12.3 is 0.1", abs(L.half_range([12.2, 12.4, 12.3, 12.3]) - 0.1) < 1e-12))
    z_lo, z_hi = L.area_extremes(5.0, 0.0, 4.0, 0.0)
    checks.append(("edge: zero error gives a single value 20", z_lo == z_hi == 20.0))
    checks.append(("worst case is wider than the typical (quadrature) error 1.49 cm^2",
                   abs(np.hypot(8.4 * 0.1, 12.3 * 0.1) - 1.489) < 1e-3 and (amax - amin) / 2 > 1.489))
    b_lo, b_hi = L.area_extremes(20.0, 0.1, 5.0, 0.1)
    checks.append(("question B: 20.0 by 5.0 gives 97.51 to 102.51, half-range 2.50, relative 2.5 %",
                   abs(b_lo - 97.51) < 5e-3 and abs(b_hi - 102.51) < 5e-3 and abs((b_hi - b_lo) / 2 - 2.5) < 5e-3
                   and abs(100 * (b_hi - b_lo) / 2 / 100.0 - 2.5) < 5e-3))
    checks.append(("question C: the area band at W = 8.4 is 102.48 to 104.16, 1.68 tall",
                   abs(L.rect_area(12.2, 8.4) - 102.48) < 5e-3 and abs(L.rect_area(12.4, 8.4) - 104.16) < 5e-3
                   and abs(L.rect_area(12.4, 8.4) - L.rect_area(12.2, 8.4) - 1.68) < 5e-3))
    h = lambda dl, dw: (lambda e: (e[1] - e[0]) / 2)(L.area_extremes(12.3, dl, 8.4, dw))
    checks.append(("question D: halving the width error gives 1.455, halving the length error 1.65 (width first)",
                   abs(h(0.1, 0.05) - 1.455) < 5e-3 and abs(h(0.05, 0.1) - 1.65) < 5e-3 and h(0.1, 0.05) < h(0.05, 0.1)))
    checks.append(("app challenge: errors of 0.04 cm on both sides give 0.83 cm^2, below 1", abs(h(0.04, 0.04) - 0.828) < 5e-3))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
